# LLM Inference Optimizer -- Colab T4 run

Runs the full V1 -> V2 -> V3 -> V4 story on a free Colab T4, using a
deliberately lightweight stack (`transformers` + `bitsandbytes`, in-process
generation -- no vLLM/llmcompressor/lm-eval) so setup takes a couple of
minutes instead of ten-plus:

- **V1** -- FP16 baseline (`eval/run_v1_baseline.py`)
- **V2** -- quantization ladder (bnb INT8/INT4) + runtime-opt sweep (batch size, context length, KV-cache, attention implementation), one variable at a time (`eval/run_v2_quant_sweep.py`)
- **V3** -- deterministic optimizer: constraint filter + Pareto frontier + manual-selection comparison (`eval/run_v3_auto_optimizer.py`)
- **V4** -- agent-assisted optimizer: diagnose -> propose -> experiment loop (`eval/run_v4_agent_optimizer.py`)
- **Report** -- before/after tables + charts (`eval/compare_v1_to_v4.py`)

**Runtime -> Change runtime type -> T4 GPU** before running anything below.

Every V is a separate cell/script so you can stop after any stage -- V3 and
the report step don't need the GPU at all, they just read whatever's in
`results/experiments.jsonl` so far.

In [ ]:
!nvidia-smi

## Setup

In [ ]:
# Replace with your actual repo URL once pushed (or use the upload-zip cell below instead)
REPO_URL = "https://github.com/sourds42/llm-inference-optimizer.git"

!git clone "$REPO_URL" llm-inference-optimizer
%cd llm-inference-optimizer

In [ ]:
# --- No repo on GitHub yet? Upload it as a zip instead ---
# from google.colab import files
# uploaded = files.upload()  # pick llm-inference-optimizer.zip
# !unzip -q llm-inference-optimizer.zip
# %cd llm-inference-optimizer

In [ ]:
# Colab's base image sometimes preinstalls a `langchain` version that
# doesn't match the langchain-core/langgraph versions we pin below, which
# crashes V4's LangGraph agent loop with
# "AttributeError: module 'langchain' has no attribute 'debug'".
# Clear those out first so the pinned versions below install cleanly.
!pip uninstall -y -q langchain langchain-core langchain-community langgraph 2>/dev/null

!pip install -q -r requirements.txt -r requirements-colab.txt
print("Installed (should be a couple of minutes -- no vllm/llmcompressor/lm-eval in this stack).")

In [ ]:
import torch

if not torch.cuda.is_available():
    print("torch.cuda.is_available() is False -- one of the installs above replaced "
          "Colab's CUDA-enabled torch with a CPU-only wheel (a known quirk when "
          "installing bitsandbytes/transformers/accelerate together). "
          "Reinstalling the CUDA build now...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "--force-reinstall",
                     "torch", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
    print("\nReinstalled. Now: Runtime -> Restart session, then re-run every cell "
          "from here down (skip the pip install cells above -- torch is already fixed).")
else:
    print(f"CUDA OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# Optional: persist results/artifacts across sessions via Drive
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/llm_inference_optimizer"
os.makedirs(DRIVE_DIR, exist_ok=True)

def save_to_drive():
    for sub in ("results", "artifacts"):
        if os.path.isdir(sub):
            dest = os.path.join(DRIVE_DIR, sub)
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(sub, dest)
    print("Saved results/ and artifacts/ to", DRIVE_DIR)

## V1 -- FP16 baseline

In [ ]:
!python -m eval.run_v1_baseline
save_to_drive()

## V2 -- quantization + runtime-opt sweep

This runs every axis in `configs/search_space.yaml` one variable at a time
off the baseline. Much smaller/faster than a vLLM-based sweep -- each config
just loads the model in-process, benchmarks with direct `generate()` calls,
and computes perplexity. One config is expected to fail honestly:
`attn_implementation=flash_attention_2` on Turing (T4) has no FA2 support --
that's captured as `error` in its row, not a crash.

In [ ]:
!python -m eval.run_v2_quant_sweep
save_to_drive()

## V3 -- deterministic optimizer

No GPU needed here -- ranks whatever's in `results/experiments.jsonl` so far.

In [ ]:
!python -m eval.run_v3_auto_optimizer

## V4 -- agent-assisted optimizer

Defaults to `MockClient` (free, deterministic). To run with a real reasoning
model instead, set one of:
- `MODEL_BACKEND=anthropic` + `ANTHROPIC_API_KEY` (paste a key below)
- `MODEL_BACKEND=ollama` -- would need Ollama installed *in this Colab VM*
  (a separate `curl -fsSL https://ollama.com/install.sh | sh` + `ollama serve &`
  + `ollama pull <model>`), not your local machine's Ollama.

In [ ]:
import os
os.environ["MODEL_BACKEND"] = "mock"  # or "anthropic" / "ollama" -- see cell above

# import getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

!python -m eval.run_v4_agent_optimizer
save_to_drive()

## Final report -- V1 -> V4 story

In [ ]:
!python -m eval.compare_v1_to_v4
save_to_drive()

In [ ]:
from IPython.display import Image, display
display(Image("results/comparison.png"))
display(Image("results/pareto.png"))